# 03 — Anomaly Detection & Data Cleaning

**Phase 2, Week 4** — CMU-Africa educational tutorial demonstrating how unsupervised AI can support energy efficiency.

**Goal:** Use Isolation Forest and DBSCAN to flag suspicious consumption patterns in smart meter data, then clean the series so Phase 3 forecasting models see a continuous 30-minute timeline.

**Scope (this notebook):** Sections 1–5 document the production baseline — load data, engineer features, train and benchmark unsupervised detectors, then interpolate flagged consumption to produce a continuity-safe dataset for Phase 3 forecasting. **Section 6** adds a research fair-comparison addendum (temporal test split) without changing the teaching path above.

**Important:** The dataset includes an `Anomaly_Label` column, but our detectors are **unsupervised** — they never use labels during training. Labels are for benchmark comparison only.

## 1. Setup & Data Loading

**Why:** Every analysis in this project starts from the validated ingestion pipeline established in Phase 1 Week 1. Reusing `load_smart_meter_data` keeps the notebook reproducible and aligned with `src/` — the same path our scripts and docs use.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.ingest_data import find_dataset_csv, get_project_root, load_smart_meter_data
from src.features.build_features import build_all_features

csv_path = find_dataset_csv(get_project_root())
df_raw = load_smart_meter_data(csv_path)

print(f"Loaded: {csv_path}")
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

## 2. Feature Engineering

**Why:** Raw consumption alone is not enough — a high reading at 2 PM may be normal while the same value at 3 AM may not. Anomaly detectors need **temporal context** (hour, day-of-week) and **rolling memory** (recent local averages). `build_all_features` applies the same pipeline used by our training scripts in `src/features/`.

In [ ]:
df = build_all_features(df_raw)

print(f"Shape after feature engineering: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

display(df.head())
display(df.tail())

## 3. Two Unsupervised Detectors

We compare two classic anomaly detectors that learn **only from feature patterns** — never from `Anomaly_Label`.

**Isolation Forest**

- Builds many random trees over the feature matrix.
- Points that are **easy to isolate** (few splits needed) are flagged as anomalies.
- Works well on high-dimensional tabular data; our default assumes ~5% contamination (`contamination=0.05`).

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**

- Groups dense neighborhoods into clusters; points in **sparse regions** are labeled noise (anomalies).
- Sensitive to hyperparameters: `eps` (neighborhood radius) and `min_samples` (minimum cluster size).
- We use the grid-search winners from Week 4 Day 2: `eps=0.5`, `min_samples=5`.

**Why both?** Different algorithms make different assumptions. Comparing them on the same benchmark helps students see trade-offs before we pick a model for data cleaning.

## 4. Train, Predict, and Benchmark

**Why:** In production we only have meter readings — no ground-truth anomaly labels. The Kaggle dataset includes `Anomaly_Label` so we can **measure** how well unsupervised detectors align with human-annotated spikes and drops.

**Evaluation rules (Phase 2 strategy):**

- Labels are mapped to `0` = Normal, `1` = Abnormal.
- We report **precision, recall, and F1** on the Abnormal class — not accuracy (the dataset is imbalanced).
- Rolling features drop the first 47 rows (warm-up window), so both models and labels are evaluated on **4,953 rows**.

We train via `detect_anomalies` — the same unified router used by our scripts — then score predictions with `evaluate_anomaly_model`.

**Research note:** This notebook uses **legacy** 15-column features and default hyperparameters (production baseline). Enhanced features, temporal train/val/test tuning, and fair legacy-vs-enhanced comparison live in `scripts/tune_*.py` and `scripts/compare_anomaly_models.py` — see [Anomaly Tuning Results](../docs/anomaly-tuning-results.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.models.evaluate_models import evaluate_anomaly_model
from src.models.train_anomaly_models import detect_anomalies, prepare_feature_matrix

sns.set_theme(style="whitegrid")

clean_index = prepare_feature_matrix(df).index
y_true = (
    df.loc[clean_index, "Anomaly_Label"]
    .map({"Normal": 0, "Abnormal": 1})
    .to_numpy(dtype=int)
)

print(f"Benchmark labels aligned: {len(y_true)} rows")
print(f"  Normal:   {(y_true == 0).sum()}")
print(f"  Abnormal: {(y_true == 1).sum()}")

In [ ]:
_, if_preds = detect_anomalies(df, model_type="isolation_forest")
_, db_preds = detect_anomalies(df, model_type="dbscan", eps=0.5, min_samples=5)

if_metrics = evaluate_anomaly_model(y_true, if_preds)
db_metrics = evaluate_anomaly_model(y_true, db_preds)

comparison = pd.DataFrame(
    [
        {
            "Model": "Isolation Forest",
            "Precision": if_metrics["precision"],
            "Recall": if_metrics["recall"],
            "F1": if_metrics["f1"],
            "Pred. anomalies": int(if_preds.sum()),
        },
        {
            "Model": "DBSCAN (eps=0.5, min_samples=5)",
            "Precision": db_metrics["precision"],
            "Recall": db_metrics["recall"],
            "F1": db_metrics["f1"],
            "Pred. anomalies": int(db_preds.sum()),
        },
    ]
)

display(comparison.round(3))

print(
    "\nTakeaway: Isolation Forest is the stronger baseline on this dataset "
    f"(F1 {if_metrics['f1']:.3f} vs {db_metrics['f1']:.3f})."
)

In [ ]:
def plot_confusion_matrix(ax, cm, title: str) -> None:
    """Plot a 2x2 confusion matrix with TN/FP/FN/TP annotations."""
    labels = np.array([["TN", "FP"], ["FN", "TP"]])
    sns.heatmap(
        cm,
        annot=labels,
        fmt="",
        cmap="Blues",
        cbar=False,
        xticklabels=["Normal", "Abnormal"],
        yticklabels=["Normal", "Abnormal"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_confusion_matrix(axes[0], if_metrics["confusion_matrix"], "Isolation Forest")
plot_confusion_matrix(
    axes[1],
    db_metrics["confusion_matrix"],
    "DBSCAN (eps=0.5, min_samples=5)",
)

plt.tight_layout()
plt.show()

## 5. Cleaning for Phase 3 — Interpolate, Don't Drop

**Why:** Phase 3 forecasters (ARIMA, LSTM, XGBoost) assume a **continuous** 30-minute timeline. If we `.drop()` rows flagged as anomalous, we create gaps — irregular intervals that break time-series models and rolling features.

**The fix:** treat suspicious consumption readings as missing values, then **time-interpolate** them. Row count stays at **5000**; only `Electricity_Consumed` values change at flagged intervals.

We use **Isolation Forest** predictions for cleaning — the stronger F1 baseline from Section 4 (0.331 vs 0.125 for DBSCAN). The same logic lives in `src/data/clean_data.py` and powers `scripts/generate_clean_data.py`.

In [ ]:
from src.data.clean_data import interpolate_anomalies

df_clean = interpolate_anomalies(df, if_preds)

changed = (df["Electricity_Consumed"] != df_clean["Electricity_Consumed"]).sum()
print(f"Shape preserved: {df_clean.shape}")
print(f"Electricity_Consumed NaNs: {df_clean['Electricity_Consumed'].isna().sum()}")
print(f"Consumption values imputed: {changed}")

In [ ]:
eval_index = prepare_feature_matrix(df).index
flagged_idx = eval_index[if_preds == 1]
center = flagged_idx[len(flagged_idx) // 2]
pos = df.index.get_loc(center)
start, end = max(0, pos - 24), min(len(df), pos + 24)

segment_orig = df.iloc[start:end]
segment_clean = df_clean.iloc[start:end]
window_flagged = segment_orig.index.isin(flagged_idx)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    segment_orig["Timestamp"],
    segment_orig["Electricity_Consumed"],
    label="Original",
    alpha=0.7,
    color="steelblue",
)
ax.plot(
    segment_clean["Timestamp"],
    segment_clean["Electricity_Consumed"],
    label="Interpolated (IF)",
    linewidth=2,
    color="darkorange",
)
ax.scatter(
    segment_orig.loc[window_flagged, "Timestamp"],
    segment_orig.loc[window_flagged, "Electricity_Consumed"],
    color="crimson",
    s=40,
    zorder=5,
    label="IF-flagged (masked)",
)
ax.set_title("Electricity consumption — original vs cleaned segment")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Electricity_Consumed")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Research Fair Comparison (Optional Addendum)

**Why:** Sections 1–5 follow the **production** path (legacy 15 features, default Isolation Forest, full-dataset scoring). Research tuning on a chronological 60/20/20 split shows how much detection improves with enhanced features and validation-tuned thresholds — without changing the default clean artifact.

Full methodology: [Anomaly Tuning Results](../docs/anomaly-tuning-results.md).

In [ ]:
from src.models.anomaly_config import TUNING_METRICS
from src.models.error_analysis import fair_comparison_table

fair_df = fair_comparison_table()
print("Fair comparison — temporal test split F1 (from anomaly_config):\n")
print(fair_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fair_df["model"], fair_df["test_f1"], color="#4C72B0")
ax.set_xlabel("Test F1")
ax.set_title("Fair head-to-head on held-out test window")
ax.set_xlim(0, 0.55)
for i, (_, row) in enumerate(fair_df.iterrows()):
    ax.text(row["test_f1"] + 0.01, i, f"{row['test_f1']:.3f}", va="center")
plt.tight_layout()
plt.show()

### How to compare uplift

| Comparison | Absolute gain | Relative gain |
|------------|---------------|---------------|
| Enhanced (0.460) vs legacy test production (0.340) | **+0.120 F1** | +35% |
| Enhanced (0.460) vs legacy test val-threshold (0.389) | **+0.071 F1** | ~18% |

Production cleaning in Section 5 still uses legacy IF. Adopting enhanced or threshold-only legacy for the Phase 3 artifact is a separate decision — see [Clean Dataset](../docs/clean-data.md#research-profiles) for optional `--profile` artifacts.

## Conclusion

This notebook walked through the full Phase 2 anomaly-detection pipeline:

1. **Load** smart meter data via the validated ingestion module
2. **Engineer features** — temporal context and rolling memory for context-aware detection
3. **Detect anomalies** unsupervised with Isolation Forest and DBSCAN
4. **Benchmark** predictions against `Anomaly_Label` (evaluation only — never used for training)
5. **Clean the series** by masking and time-interpolating flagged consumption

To generate the Phase 3 artifact locally:

```bash
python scripts/generate_clean_data.py
```

This writes `data/processed/clean_smart_meter_data.csv` (5000 × 15, 0 NaNs). The file is gitignored — regenerate on each machine.

**Phase 2 is complete.** We now have a continuity-safe, feature-engineered dataset ready for Phase 3 time-series forecasting.